[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/11_kosten_recht_datenschutz.ipynb)

# Sitzung 11 — Kosten, Latenz, Recht & Datenschutz

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI · 16.12.2026**

Unser Review Radar läuft: 279 Bewertungen, ein paar Cent, eine Minute. Lena ist begeistert und fragt: *„Können wir das für alle unsere Produkte machen? Jeden Monat?"* Plötzlich sind es nicht 279, sondern 100.000 Bewertungen. Und ihr Datenschutzbeauftragter fragt: *„Moment, wohin schickt ihr eigentlich die Kundentexte?"*

Heute beantworten wir zwei Fragen, die jedes echte Projekt stellen muss:

1. **Was würde es kosten, und wie lange dauert es, im großen Maßstab?**
2. **Was ist rechtlich zu klären, bevor man es tut?**

**Ablauf**
1. **Was kostet eine Bewertung? (Abschnitt 1):** Tokens, Preise, eine echte Messung.
2. **Hochrechnen (Abschnitt 2):** von 279 auf eine Million, und die Hebel, die die Kosten senken.
3. **Latenz und Rate-Limits (Abschnitt 3):** wie lange es dauert, und wo die Grenze liegt.
4. **Lokal oder Cloud? (Abschnitt 4):** ein kurzer Blick zurück auf Sitzung 5.
5. **Darf ich diese Daten nutzen? (Abschnitt 5):** Lizenzen, am echten Fall dieses Kurses.
6. **Darf ich diese Daten verschicken? (Abschnitt 6):** DSGVO, wenn Kundentexte an eine US-API gehen.
7. **Eure Aufgabe (Abschnitt 7):** Kosten und Rechtsfragen für euer eigenes Projekt.
8. **Der Bericht (Abschnitt 8):** Was würde es kosten, und was ist zu klären?

> ⚖️ **Wichtig vorab:** Die Abschnitte 5 und 6 sind **keine Rechtsberatung**. Ihr lernt, welche **Fragen** man stellen muss und wo man nachschaut. Ob ein konkretes Projekt zulässig ist, entscheiden Fachleute: an der HTW und in Unternehmen ist die erste Ansprechperson der oder die **Datenschutzbeauftragte**.

### Rückblick auf Sitzung 10

```text
1. Fake-Bewertungen                    -> verschieben Aggregate, obwohl jede einzelne „gültig" aussieht
2. Sarkasmus                           -> „Ganz großes Kino, 40 Minuten Akku": positive Wörter, negative Aussage
3. Englische Bewertungen               -> ein Prompt, zwei Sprachen: funktioniert, muss aber gemessen werden
4. Saubere Vergleichsdaten             -> eine kleine, saubere Stichprobe zeigt, wie viel der Schmutz verändert
5. Wer schreibt Bewertungen?           -> nicht „die Kunden", sondern die, die etwas zu sagen haben (Selektion)
```

Sitzung 10 hat gezeigt: Die **Daten** entscheiden oft mehr über das Ergebnis als das Modell. Heute geht es um zwei weitere Dinge, die über Erfolg entscheiden, bevor das Modell überhaupt rechnet: **Geld und Recht**.

## 0. Setup und Daten

**Was passiert in der nächsten Zelle?** Bibliotheken laden, `radar.py` holen, das Modell einstellen, die bereinigten Bewertungen laden. Mit Schlüssel messen wir heute echte Aufrufe (etwa 40, Bruchteile eines Cents). Ohne Schlüssel simuliert der Mock eine plausible Wartezeit, die Token-Zahlen sind dann grob geschätzt.

In [ ]:
!pip install anthropic --quiet

import json, re, time, urllib.request, importlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
pd.set_option("display.max_colwidth", 90)

URL = "https://raw.githubusercontent.com/kevisback/bda-course/main/wise2627/notebooks/radar.py"
try:
    urllib.request.urlretrieve(URL, "radar.py")
except Exception as e:
    print("Download fehlgeschlagen, nutze lokale radar.py:", e)
import radar
importlib.reload(radar)
radar.setup(backend="anthropic", model="claude-haiku-4-5")

df, wahrheit = radar.lade_sauber()
print(df.shape)

**Wie lese ich das?** Dieselben 279 Bewertungen wie seit Sitzung 4. Heute interessiert uns nicht, *was* in ihnen steht, sondern was es **kostet**, sie zu analysieren, und ob wir es **dürfen**.

## 1. Was kostet eine Bewertung?

Ein LLM rechnet nicht in Wörtern, sondern in **Tokens**: Wortstücken, im Deutschen grob drei bis vier Buchstaben. Bezahlt wird pro Million Tokens, getrennt nach dem, was wir **hinschicken** (Input) und was zurückkommt (Output). Für Claude Haiku 4.5 sind das laut Preisliste von Anthropic ([Quelle](https://platform.claude.com/docs/en/about-claude/pricing), Stand November 2026):

```text
Input:   1 USD pro Million Tokens
Output:  5 USD pro Million Tokens      <- Output ist fünfmal so teuer
```

Ein Rechenbeispiel mit runden Zahlen: Eine Analyse mit 500 Input- und 100 Output-Tokens kostet

```text
500 × 1 USD / 1.000.000  +  100 × 5 USD / 1.000.000  =  0,0005 + 0,0005  =  0,001 USD
```

Ein Zehntel Cent pro Bewertung. Klingt nach nichts, aber mal 100.000 sind es 100 USD.

**Was passiert in der nächsten Zelle?** Dieselbe Rechnung als Funktion, mit den Preisen aus `radar.py`.

In [ ]:
PREIS_IN, PREIS_OUT = radar.PREIS_INPUT_PRO_MTOK, radar.PREIS_OUTPUT_PRO_MTOK   # USD pro Million Tokens

def kosten(anzahl, tok_in, tok_out, preis_in=PREIS_IN, preis_out=PREIS_OUT):
    """Kosten in USD für `anzahl` Aufrufe mit je tok_in Input- und tok_out Output-Tokens."""
    return anzahl * (tok_in * preis_in + tok_out * preis_out) / 1_000_000

print(f"Preise: {PREIS_IN} USD (Input) und {PREIS_OUT} USD (Output) pro Million Tokens")
print(f"Beispiel, 1 Aufruf mit 500 / 100 Tokens:       {kosten(1, 500, 100):.4f} USD")
print(f"Beispiel, 100.000 Aufrufe mit 500 / 100 Tokens: {kosten(100_000, 500, 100):,.0f} USD")

**Wie lese ich das?** Die Funktion ist der ganze „Kostenrechner": Anzahl × Tokens × Preis. Alles Weitere heute ist Einsetzen von Zahlen. Die eigentliche Arbeit ist, **gute Zahlen** für die Tokens zu bekommen, und die misst man am besten.

**Was passiert in der nächsten Zelle?** Wir analysieren 20 Bewertungen mit dem Prompt aus Sitzung 8, **nacheinander**, und messen für jeden Aufruf: Input-Tokens, Output-Tokens, Wartezeit.

In [ ]:
ANALYSE_PROMPT = """Du analysierst eine Produktbewertung zu kabellosen Ohrhörern.

Antworte ausschließlich mit einem JSON-Objekt mit genau diesen Feldern:
- "sentiment": "positive", "negative", "mixed" oder "neutral"
- "themen": Liste von {"thema": <Thema>, "polaritaet": "positiv" oder "negativ"}
   Erlaubte Themen: """ + ", ".join(radar.THEMEN) + """
- "begruendung": höchstens 20 Wörter
- "dringend": true, wenn ein Defekt oder ein Problem gemeldet wird, um das sich jemand kümmern sollte, sonst false"""

def ein_aufruf(text):
    """Ein Analyse-Aufruf; gibt Tokens und Wartezeit zurück."""
    start = time.time()
    antwort = radar.mit_retry(radar.call_llm, f'Bewertung:\n"""{text}"""', system=ANALYSE_PROMPT, max_tokens=400)
    return {"text": text, "input_tokens": antwort["input_tokens"], "output_tokens": antwort["output_tokens"],
            "sekunden": time.time() - start}

probe = df["text"].sample(20, random_state=11).tolist()
ECHT = radar.BACKEND == "anthropic"
if ECHT:
    messung = pd.DataFrame([ein_aufruf(t) for t in probe])
else:                                                     # Mock: plausible Wartezeit simulieren
    with radar.simulation(fehlerrate=0, latenz=0.8):
        messung = pd.DataFrame([ein_aufruf(t) for t in probe])
    print("(Mock: Wartezeit simuliert, Tokens grob geschätzt)\n")

TOK_IN = messung["input_tokens"].mean()
TOK_OUT = messung["output_tokens"].mean()
LATENZ = messung["sekunden"].mean()
print(f"Mittel pro Aufruf: {TOK_IN:.0f} Input-Tokens, {TOK_OUT:.0f} Output-Tokens, {LATENZ:.2f} Sekunden")
print(f"Kosten pro Bewertung: {kosten(1, TOK_IN, TOK_OUT):.5f} USD")
messung[["input_tokens", "output_tokens", "sekunden"]].describe().round(2).loc[["mean", "min", "max"]]

**Wie lese ich das?** Drei Zahlen merken wir uns für den Rest der Sitzung: Input-Tokens, Output-Tokens und Sekunden pro Aufruf (`TOK_IN`, `TOK_OUT`, `LATENZ`). Achtet auf zwei Dinge:
- Die **Input-Tokens** schwanken kaum. Fast alles davon ist der Prompt, und der ist bei jedem Aufruf derselbe.
- Die **Wartezeit** schwankt deutlich. Manche Aufrufe brauchen doppelt so lang wie andere, je nach Auslastung beim Anbieter.

**Was passiert in der nächsten Zelle?** Wofür bezahlen wir eigentlich? Wir schätzen, welcher Teil der Input-Tokens auf die Bewertung entfällt und welcher auf den Prompt (grob: vier Zeichen pro Token).

In [ ]:
tok_bewertung = messung["text"].str.len().mean() / 4          # grobe Schätzung: ~4 Zeichen pro Token
tok_prompt = TOK_IN - tok_bewertung

teile = pd.Series({"Prompt (Input)": tok_prompt * PREIS_IN, "Bewertung (Input)": tok_bewertung * PREIS_IN,
                   "Antwort (Output)": TOK_OUT * PREIS_OUT}) / 1_000_000
anteile = teile / teile.sum()

fig, ax = plt.subplots(figsize=(8, 1.8))
links = 0
for (name, wert), farbe in zip(anteile.items(), ["#9ca3af", "#2563eb", "#f59e0b"]):
    ax.barh([0], [wert], left=links, color=farbe)
    if wert >= 0.12:
        ax.text(links + wert / 2, 0, f"{name}\n{wert:.0%}", ha="center", va="center", fontsize=9, color="white")
    else:                                                  # schmales Stück: Beschriftung darunter
        ax.annotate(f"{name} {wert:.0%}", (links + wert / 2, -0.45), ha="center", va="top", fontsize=9, color=farbe,
                    annotation_clip=False)
    links += wert
ax.set_xlim(0, 1); ax.axis("off"); ax.set_title("Wofür bezahlen wir pro Bewertung?", loc="left")
plt.tight_layout(); plt.show()
print(f"Prompt ~{tok_prompt:.0f} Tokens, Bewertung ~{tok_bewertung:.0f} Tokens, Antwort ~{TOK_OUT:.0f} Tokens")

**Wie lese ich das?** Die Bewertung selbst ist der **kleinste** Posten. Wir bezahlen vor allem für zwei Dinge: den Prompt, der bei jedem Aufruf mitgeschickt wird, und die Antwort, deren Tokens fünfmal so viel kosten. Das ist die wichtigste Einsicht für alle Sparmaßnahmen gleich: **Gespart wird beim Prompt und bei der Antwort, nicht bei den Daten.**

## 2. Hochrechnen

**Was passiert in der nächsten Zelle?** Wir setzen die gemessenen Werte in die Kostenfunktion ein: für unsere 279 Bewertungen, für 10.000, 100.000 und eine Million.

In [ ]:
mengen = [len(df), 10_000, 100_000, 1_000_000]
hochrechnung = pd.DataFrame({"Bewertungen": mengen,
                             "Kosten (USD)": [kosten(n, TOK_IN, TOK_OUT) for n in mengen],
                             "Tokens gesamt (Mio.)": [n * (TOK_IN + TOK_OUT) / 1e6 for n in mengen]})

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar([f"{n:,}".replace(",", ".") for n in mengen], hochrechnung["Kosten (USD)"], color="#2563eb")
ax.set_yscale("log"); ax.set_ylabel("USD (logarithmisch)")
for i, k in enumerate(hochrechnung["Kosten (USD)"]):
    ax.annotate(f"{k:,.2f} USD".replace(",", "X").replace(".", ",").replace("X", "."), (i, k), ha="center",
                xytext=(0, 4), textcoords="offset points", fontsize=9)
ax.set_title("Kosten der Analyse nach Anzahl der Bewertungen"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
hochrechnung.style.format({"Kosten (USD)": "{:,.2f}", "Tokens gesamt (Mio.)": "{:,.1f}"})

**Wie lese ich das?** Die Kosten wachsen genau **linear**: zehnmal so viele Bewertungen, zehnmal so viel Geld. Die y-Achse ist logarithmisch, sonst wären die ersten Balken unsichtbar. Für ein Unternehmen ist selbst die Million meist bezahlbar. Teuer wird es erst, wenn man das **wiederholt**: jeden Monat neu, für jedes Produkt, und jedes Mal, wenn jemand den Prompt verbessert und alles neu analysiert.

> **Nicht:** „Pro Bewertung kostet das nichts."
> **Sondern:** „Pro Bewertung kostet es wenig; entscheidend ist, wie oft wir **alles** neu rechnen."

✏️ **Eure Aufgabe 1: Was kostet Lenas Wunsch?**

Lena will den Radar für alle **40 Produkte** des Unternehmens. Pro Produkt kommen im Schnitt **2.500 Bewertungen pro Jahr** dazu, und es gibt schon **drei Jahre** Bewertungen im Bestand.

1. Was kostet die **einmalige** Analyse des Bestands?
2. Was kosten die **laufenden** Kosten pro Monat für neue Bewertungen?
3. Das Team will den Prompt in einem Jahr **viermal** verbessern und den Bestand jedes Mal neu analysieren. Was kostet das zusätzlich?

Füllt die Lücken `...` und nutzt `kosten(anzahl, TOK_IN, TOK_OUT)`. Die Lösung steht in der Zelle danach.

In [ ]:
# Eure Lösung: Lücken ... füllen
# (try/except nur, damit „Alle ausführen" hier nicht stoppt. Die Lösungszelle darunter braucht es nicht.)
try:
    bestand = 40 * 2500 * ...                         # Bewertungen aus drei Jahren
    neu_pro_monat = ...                               # 40 Produkte, 2.500 pro Jahr, durch 12 Monate
    einmalig = kosten(bestand, TOK_IN, TOK_OUT)
    pro_monat = kosten(..., TOK_IN, TOK_OUT)
    neu_rechnen = 4 * ...
    print(f"Bestand: {bestand:,} Bewertungen, einmalig {einmalig:,.2f} USD")
    print(f"Laufend: {neu_pro_monat:,.0f} Bewertungen pro Monat, {pro_monat:,.2f} USD pro Monat")
    print(f"Vier Prompt-Verbesserungen: {neu_rechnen:,.2f} USD")
except Exception as e:
    print("Noch nicht fertig oder ein Fehler:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
bestand = 40 * 2500 * 3
neu_pro_monat = 40 * 2500 / 12
einmalig = kosten(bestand, TOK_IN, TOK_OUT)
pro_monat = kosten(neu_pro_monat, TOK_IN, TOK_OUT)
neu_rechnen = 4 * einmalig

print(f"Bestand: {bestand:,} Bewertungen, einmalig {einmalig:,.2f} USD")
print(f"Laufend: {neu_pro_monat:,.0f} Bewertungen pro Monat, {pro_monat:,.2f} USD pro Monat ({12 * pro_monat:,.2f} USD pro Jahr)")
print(f"Vier Prompt-Verbesserungen: {neu_rechnen:,.2f} USD")
print(f"\nErstes Jahr insgesamt: {einmalig + 12 * pro_monat + neu_rechnen:,.2f} USD")

**Wie lese ich das?** 300.000 Bewertungen im Bestand, gut 8.000 neue pro Monat. Die **laufenden** Kosten sind klein. Der größte Posten im ersten Jahr ist das **Neu-Rechnen** nach Prompt-Änderungen: Jede Verbesserung kostet so viel wie die erste Analyse. Deshalb testet man einen neuen Prompt zuerst an einer **Stichprobe** (Sitzung 3) und rechnet erst dann alles neu, wenn er nachweislich besser ist.

Und eine Zahl fehlt in dieser Rechnung ganz: **Arbeitszeit**. Ein Tag Entwicklung kostet ein Unternehmen mehr als die gesamte API-Rechnung oben.

### Die Hebel: Wie wird es billiger?

Aus Abschnitt 1 wissen wir, wofür wir bezahlen: vor allem Prompt und Antwort. Daraus ergeben sich vier Hebel:

| Hebel | Was passiert | Preis dafür |
|---|---|---|
| **Kürzerer Prompt** | weniger Input-Tokens pro Aufruf | Qualität muss neu gemessen werden |
| **Mehrere Bewertungen pro Aufruf** (Batching, Sitzung 3) | der Prompt wird nur einmal pro 20 Bewertungen bezahlt | Lücken in langen Antworten (Sitzung 3 und 5) |
| **Batch-API des Anbieters** | 50 % Rabatt auf alle Tokens ([Quelle](https://platform.claude.com/docs/en/build-with-claude/batch-processing)) | Ergebnisse kommen nicht sofort, sondern meist innerhalb einer Stunde, spätestens nach 24 Stunden |
| **Kürzere Antwort** | z. B. Begründung weglassen | weniger Nachvollziehbarkeit (Sitzung 6) |

**Was passiert in der nächsten Zelle?** Wir rechnen jeden Hebel für 100.000 Bewertungen durch. Die Annahmen stehen im Code: Batching teilt den Prompt auf 20 Bewertungen auf, eine Antwort ohne Begründung ist halb so lang.

In [ ]:
N = 100_000
varianten = {
    "Basis (wie gemessen)":        kosten(N, TOK_IN, TOK_OUT),
    "Prompt halb so lang":         kosten(N, tok_prompt / 2 + tok_bewertung, TOK_OUT),
    "Antwort halb so lang":        kosten(N, TOK_IN, TOK_OUT / 2),
    "20 Bewertungen pro Aufruf":   kosten(N, tok_prompt / 20 + tok_bewertung, TOK_OUT),
    "Batch-API (−50 %)":           kosten(N, TOK_IN, TOK_OUT) * 0.5,
    "20 pro Aufruf + Batch-API":   kosten(N, tok_prompt / 20 + tok_bewertung, TOK_OUT) * 0.5,
}
hebel = pd.Series(varianten)

fig, ax = plt.subplots(figsize=(8, 3.6))
farben = ["#9ca3af"] + ["#2563eb"] * (len(hebel) - 1)
ax.barh(hebel.index[::-1], hebel.values[::-1], color=farben[::-1])
for i, v in enumerate(hebel.values[::-1]):
    ax.annotate(f"{v:,.0f} USD", (v, i), xytext=(3, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_title(f"Kosten für {N:,} Bewertungen je nach Hebel".replace(",", "."))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
(hebel / hebel.iloc[0]).map(lambda v: f"{v:.0%} der Basis")

**Wie lese ich das?** Die Hebel wirken sehr unterschiedlich. **Mehrere Bewertungen pro Aufruf** spart am meisten Input, weil der Prompt nur noch einmal pro 20 Bewertungen bezahlt wird; dafür steigt das Risiko von Lücken, das wir in Sitzung 3 gesehen haben. Die **Batch-API** halbiert alles, wenn man auf die Ergebnisse warten kann. Für einen monatlichen Bericht ist das kein Problem, für ein Live-Dashboard schon.

Jeder Hebel hat einen Preis, und der heißt fast immer **Qualität oder Geschwindigkeit**. Deshalb gehört zu jeder Sparmaßnahme eine neue Messung wie in Sitzung 8.

> **Nicht:** „Wir nehmen das billigste Modell, dann wird es günstig."
> **Sondern:** Erst messen, wofür die Tokens draufgehen; dann den Hebel wählen, der am meisten spart und die Qualität nachweislich hält.

## 3. Latenz und Rate-Limits

Geld ist das eine, **Zeit** das andere. In Abschnitt 1 haben wir gemessen, wie lange ein Aufruf dauert (`LATENZ`).

**Was passiert in der nächsten Zelle?** Wie lange würde es dauern, alle Bewertungen **nacheinander** zu analysieren?

In [ ]:
def dauer_text(sekunden):
    if sekunden < 120:
        return f"{sekunden:.0f} Sekunden"
    if sekunden < 2 * 3600:
        return f"{sekunden / 60:.0f} Minuten"
    if sekunden < 2 * 86400:
        return f"{sekunden / 3600:.1f} Stunden"
    return f"{sekunden / 86400:.1f} Tage"

for n in [len(df), 10_000, 100_000, 1_000_000]:
    print(f"{n:>9,} Bewertungen nacheinander:  {dauer_text(n * LATENZ)}".replace(",", "."))

**Wie lese ich das?** Nacheinander ist schon bei 100.000 Bewertungen unbrauchbar: Das wären viele Stunden bis Tage, in denen ein einzelnes Programm wartet. Fast die ganze Zeit wartet es übrigens nur auf die Antwort; der eigene Rechner tut dabei nichts.

**Was passiert in der nächsten Zelle?** Wir schicken dieselben 20 Bewertungen **parallel**, acht gleichzeitig (Sitzung 3), und vergleichen die Gesamtzeit.

In [ ]:
start = time.time()
if ECHT:
    with ThreadPoolExecutor(max_workers=8) as pool:
        parallel = list(pool.map(ein_aufruf, probe))
else:
    with radar.simulation(fehlerrate=0, latenz=0.8):
        with ThreadPoolExecutor(max_workers=8) as pool:
            parallel = list(pool.map(ein_aufruf, probe))
zeit_parallel = time.time() - start
zeit_nacheinander = messung["sekunden"].sum()

print(f"20 Aufrufe nacheinander: {zeit_nacheinander:.1f} Sekunden")
print(f"20 Aufrufe, 8 parallel:  {zeit_parallel:.1f} Sekunden")
print(f"Beschleunigung:          {zeit_nacheinander / zeit_parallel:.1f}-fach")

**Wie lese ich das?** Mit acht gleichzeitigen Anfragen geht es mehrfach schneller, aber nicht ganz achtmal: Bei 20 Aufrufen wartet die letzte Runde auf die langsamsten Antworten. Bei 100.000 Aufrufen spielt dieser Rest keine Rolle mehr. Dann gilt fast genau: **acht parallel = achtmal so schnell.**

Warum nicht gleich 1.000 parallel? Weil der Anbieter das nicht erlaubt.

### Rate-Limits: die Obergrenze

Jeder API-Schlüssel hat Grenzen, wie viel er pro Minute schicken darf. Unser Kurs-Schlüssel hat laut Anthropic Console drei:

```text
Anfragen pro Minute         (RPM)     20.000
Input-Tokens pro Minute     (ITPM)    10.000.000
Output-Tokens pro Minute    (OTPM)     2.000.000
```

Welche Grenze zuerst greift, hängt davon ab, wie viele Tokens eine Anfrage hat. Andere Schlüssel, Konten und Modelle haben andere Grenzen; für ein echtes Projekt schaut man in der eigenen Console nach.

**Was passiert in der nächsten Zelle?** Wir rechnen für jede der drei Grenzen aus, wie viele Analysen pro Minute sie erlaubt, und welche davon der **Engpass** ist.

In [ ]:
LIMITS = {"Anfragen pro Minute": 20_000, "Input-Tokens pro Minute": 10_000_000, "Output-Tokens pro Minute": 2_000_000}

erlaubt = pd.Series({
    "Anfragen pro Minute": LIMITS["Anfragen pro Minute"],
    "Input-Tokens pro Minute": LIMITS["Input-Tokens pro Minute"] / TOK_IN,
    "Output-Tokens pro Minute": LIMITS["Output-Tokens pro Minute"] / TOK_OUT,
}).rename("Analysen pro Minute")
engpass = erlaubt.idxmin()
max_pro_minute = erlaubt.min()

# Little's Law (Sitzung 3): gleichzeitige Anfragen = Durchsatz pro Sekunde × Wartezeit
noetig_parallel = max_pro_minute / 60 * LATENZ

display(erlaubt.map(lambda v: f"{v:,.0f}".replace(",", ".")).to_frame())
print(f"Engpass: {engpass} -> höchstens {max_pro_minute:,.0f} Analysen pro Minute".replace(",", "."))
print(f"Dafür nötig: etwa {noetig_parallel:,.0f} gleichzeitige Anfragen".replace(",", "."))
print(f"100.000 Bewertungen am Limit: {dauer_text(100_000 / max_pro_minute * 60)}")

**Wie lese ich das?** Jede Grenze erlaubt eine bestimmte Zahl von Analysen pro Minute, und die **kleinste** davon gilt. Welche das ist, hängt vom Verhältnis der Tokens ab: Das Output-Kontingent ist fünfmal kleiner als das Input-Kontingent. Werden die Antworten länger, etwa durch eine ausführlichere Begründung, wird schnell die Output-Grenze zum Engpass (Zum Weiterprobieren, Punkt 3).

Die zweite Zahl kommt aus **Little's Law** (Sitzung 3): Um diesen Durchsatz zu erreichen, müssten so viele Anfragen gleichzeitig unterwegs sein. Das sind deutlich mehr als acht, und genau deshalb ist am Limit 100.000 Bewertungen eine Sache von Minuten, nicht von Tagen.

**Was passiert in der nächsten Zelle?** Das Ganze als Bild: Wie viele Analysen pro Minute schaffen wir mit wie vielen gleichzeitigen Anfragen, und wo deckelt die Grenze?

In [ ]:
parallel_werte = np.arange(1, int(noetig_parallel * 2) + 2)
ohne_grenze = parallel_werte / LATENZ * 60                    # Analysen pro Minute ohne Limit
mit_grenze = np.minimum(ohne_grenze, max_pro_minute)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(parallel_werte, ohne_grenze, ls="--", color="#9ca3af", label="ohne Grenze")
ax.plot(parallel_werte, mit_grenze, color="#2563eb", lw=2.5, label="mit Rate-Limit")
ax.axhline(max_pro_minute, color="#dc2626", lw=1)
ax.annotate(f"Grenze: {engpass}", (parallel_werte[1], max_pro_minute * 1.03), color="#dc2626", fontsize=9)
ax.axvline(8, color="#111827", ls=":", lw=1)
ax.annotate("8 parallel", (8, max_pro_minute * 0.55), xytext=(6, 0), textcoords="offset points", fontsize=9)
ax.set_xlabel("gleichzeitige Anfragen"); ax.set_ylabel("Analysen pro Minute")
ax.set_title("Mehr parallel hilft, bis das Rate-Limit greift"); ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Links steigt die Linie gerade an: doppelt so viele gleichzeitige Anfragen, doppelt so viele Analysen pro Minute. Dann knickt sie ab, an der roten Grenze. Mehr Parallelität bringt ab dort nichts mehr; der Anbieter antwortet mit Fehler 429 („zu viele Anfragen"), und unser Retry aus Sitzung 3 wartet.

Mit acht parallelen Anfragen (gepunktete Linie) sind wir noch weit unter der Grenze. Für unseren Kurs genügt das völlig.

> 💡 **Kernpunkt: Überschlagsrechnung vor dem Bau.** Drei gemessene Zahlen (Tokens rein, Tokens raus, Sekunden) und drei Grenzen aus der Console reichen, um vor dem ersten großen Lauf zu sagen: *Was kostet es? Wie lange dauert es? Wo ist der Engpass?*
>
> **Nicht:** „Wir starten mal und schauen, was die Rechnung sagt."
> **Sondern:** Vorher 20 Aufrufe messen, hochrechnen, dann entscheiden.

## 4. Lokal oder Cloud?

In Sitzung 5 lief das Embedding-Modell **lokal** in Colab: kein Schlüssel, keine Kosten pro Aufruf, die Texte gingen an keinen LLM-Anbieter. Warum nicht auch die Analyse lokal?

| | Cloud-API (z. B. Haiku) | eigenes Modell auf eigenem Server |
|---|---|---|
| Kosten | pro Token, wächst mit der Menge | fester Betrag pro Monat (Server, Strom, Betreuung) |
| Start | sofort, ein Schlüssel genügt | Server, Modell, Einrichtung, Wartung |
| Qualität | große, aktuelle Modelle | meist kleinere Modelle; Qualität muss gemessen werden |
| Daten | verlassen das Haus (Abschnitt 6) | bleiben im eigenen Netz |
| Grenzen | Rate-Limits des Anbieters | Rechenleistung des eigenen Servers |

**Was passiert in der nächsten Zelle?** Eine Überschlagsrechnung für die Kostenfrage: Ab welcher Menge pro Monat wäre ein eigener Server billiger? Die Serverkosten sind eine **Annahme** zum Ausprobieren, kein Marktpreis; ändert sie in der ersten Zeile.

In [ ]:
SERVER_PRO_MONAT = 1500          # ANNAHME in USD: Miete, Strom, Betreuung zusammen. Bitte variieren!

monatsmengen = np.logspace(3, 8, 200)                      # 1.000 bis 100 Mio. Bewertungen pro Monat
api = kosten(monatsmengen, TOK_IN, TOK_OUT)
gleichstand = SERVER_PRO_MONAT / kosten(1, TOK_IN, TOK_OUT)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.loglog(monatsmengen, api, color="#2563eb", lw=2, label="Cloud-API (pro Token)")
ax.axhline(SERVER_PRO_MONAT, color="#16a34a", lw=2, label="eigener Server (fest, Annahme)")
ax.axvline(gleichstand, color="#111827", ls=":", lw=1)
ax.annotate(f"Gleichstand bei\n~{round(gleichstand, -4):,.0f} Bewertungen/Monat".replace(",", "."), (gleichstand * 1.2, SERVER_PRO_MONAT * 0.05), fontsize=9)
ax.scatter([neu_pro_monat], [pro_monat], color="#dc2626", zorder=3)
ax.annotate("Lenas Fall", (neu_pro_monat, pro_monat), xytext=(6, -12), textcoords="offset points", fontsize=9, color="#dc2626")
ax.set_xlabel("Bewertungen pro Monat"); ax.set_ylabel("USD pro Monat")
ax.set_title("Ab wann lohnt sich ein eigener Server? (nur Kosten)"); ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
print(f"Lenas Fall: {neu_pro_monat:,.0f} neue Bewertungen pro Monat kosten über die API {pro_monat:,.2f} USD".replace(",", "."))

**Wie lese ich das?** Bei kleinen Mengen ist die API viel billiger, bei sehr großen ein eigener Server. Lenas Fall liegt weit links vom Gleichstand: Für ein paar Tausend Bewertungen im Monat lohnt sich kein eigener Server, schon gar nicht, wenn man die Arbeitszeit für Einrichtung und Wartung mitrechnet.

Kosten sind aber selten der Grund für „lokal". Die häufigeren Gründe stehen in der Tabelle oben in der Zeile **Daten**: Manche Daten *dürfen* das Haus nicht verlassen. Damit sind wir beim Recht.

## 5. Darf ich diese Daten nutzen?

Eine Geschichte aus der Vorbereitung dieses Kurses. Im September 2026 war der Plan, mit **echten Amazon-Bewertungen** zu arbeiten. Es gibt dafür bekannte, oft zitierte Datensätze, etwa das *Multilingual Amazon Reviews Corpus*, auch auf Deutsch. Ein Blick in die Nutzungsbedingungen ([Quelle](https://docs.opendata.aws/amazon-reviews-ml/readme.html)) zeigt:

- Die Nutzung ist erlaubt **„for purposes of academic research"**, also für akademische Forschung,
- die Lizenz ist **nicht übertragbar** und darf nicht weiterlizenziert werden,
- **kommerzielle Nutzung ist ausgeschlossen**, ausdrücklich auch Forschung, die mit Finanzierung, Beratung oder Ergebnissen für Unternehmen verbunden ist,
- **weitergeben oder veröffentlichen** ist nicht erlaubt.

Für diesen Kurs hieß das:
- Unser Material liegt in einem **öffentlichen GitHub-Repository**. Die Daten dort abzulegen, wäre eine Weitergabe gewesen.
- Der Kurs ist ein **bezahlter Lehrauftrag**, und der Dozent arbeitet auch für ein Unternehmen. Ob das noch „akademische Forschung" ist, ist zumindest unklar.

Die Entscheidung: **synthetische Daten.** Die Nimbus Q2 ist erfunden, jede Bewertung erzeugt `radar.py` selbst. Dafür gibt es keine fremden Rechte. Unser eigenes Material steht unter offenen Lizenzen: der Code unter MIT, die Lehrmaterialien unter CC BY 4.0 ([Quelle](https://creativecommons.org/licenses/by/4.0/)). Jeder darf es weiterverwenden, wenn er die Quelle nennt.

**Was passiert in der nächsten Zelle?** Eine „Lizenz-Ampel" als einfache Faustregel für vier typische Datenquellen. Sie ersetzt keine Prüfung, sie zeigt nur, **welche Fragen** man stellen muss.

In [ ]:
QUELLEN = {
    "Amazon Reviews Corpus (Forschungslizenz)": {"eigene_daten": False, "lizenz_bekannt": True,
        "kommerziell_erlaubt": False, "weitergabe_erlaubt": False, "personenbezug_moeglich": True},
    "Unser synthetischer Datensatz (MIT / CC BY 4.0)": {"eigene_daten": True, "lizenz_bekannt": True,
        "kommerziell_erlaubt": True, "weitergabe_erlaubt": True, "personenbezug_moeglich": False},
    "Bewertungen von fremder Website kopiert (Scraping)": {"eigene_daten": False, "lizenz_bekannt": False,
        "kommerziell_erlaubt": None, "weitergabe_erlaubt": None, "personenbezug_moeglich": True},
    "Bewertungen aus dem eigenen Shop": {"eigene_daten": True, "lizenz_bekannt": True,
        "kommerziell_erlaubt": True, "weitergabe_erlaubt": None, "personenbezug_moeglich": True},
}

def ampel(q):
    """Faustregel, keine Rechtsprüfung: rot = so nicht, gelb = Fragen klären, grün = unkritisch."""
    if q["lizenz_bekannt"] and q["kommerziell_erlaubt"] is False:
        return "🔴 nur im erlaubten Rahmen (z. B. Forschung), nicht weitergeben"
    if not q["lizenz_bekannt"]:
        return "🟡 Rechte unklar: Nutzungsbedingungen der Quelle prüfen"
    if q["personenbezug_moeglich"]:
        return "🟡 Nutzung wohl möglich, aber Datenschutz klären (Abschnitt 6)"
    return "🟢 unkritisch"

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({"Einschätzung": {name: ampel(q) for name, q in QUELLEN.items()}}))

**Wie lese ich das?** Nur eine Quelle ist grün, und das ist die, die wir selbst erzeugt haben. Zwei Beobachtungen:
- **„Öffentlich sichtbar" heißt nicht „frei nutzbar".** Bewertungen auf einer fremden Website kann jeder lesen. Ob man sie kopieren, speichern und auswerten darf, regeln die Nutzungsbedingungen der Website und das Urheber- und Datenbankrecht. Das muss man prüfen, nicht annehmen.
- **Auch eigene Daten sind nicht automatisch unkritisch.** Die Bewertungen aus dem eigenen Shop gehören zwar dem Unternehmen, sie wurden aber von Menschen geschrieben. Damit sind wir beim Datenschutz.

Fragen, die ihr an **jede** Datenquelle stellen solltet:

```text
1. Wer hat die Rechte an den Daten?            (wir selbst / ein Anbieter / unklar)
2. Unter welchen Bedingungen stehen sie?       (Lizenz, Nutzungsbedingungen, Vertrag)
3. Ist mein Zweck erlaubt?                     (Forschung / Lehre / kommerziell)
4. Darf ich sie speichern und weitergeben?     (z. B. in ein öffentliches Repository)
5. Darf ich sie an einen KI-Dienst schicken?   (manche Bedingungen schließen das aus)
```

## 6. Darf ich diese Daten verschicken?

Jedes Mal, wenn `radar.call_llm` läuft, geht eine Kundenbewertung an einen Server von Anthropic, einem US-Unternehmen. Das ist der Punkt, an dem die **Datenschutz-Grundverordnung** (DSGVO, [Text bei EUR-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj/deu)) ins Spiel kommt.

```text
Kundin  ──>  Shop  ──>  unsere Pipeline  ──>  LLM-API (USA)  ──>  Antwort  ──>  Dashboard
  │           │              │                      │
  schreibt    speichert      schickt den            verarbeitet den Text,
  Bewertung   Bewertung      Text weiter            je nach Vertrag kurz gespeichert
```

**Sind Bewertungen überhaupt personenbezogene Daten?** Personenbezogen sind nach Art. 4 Nr. 1 DSGVO alle Informationen, die sich auf eine identifizierte oder **identifizierbare** Person beziehen. Eine Bewertung allein ist oft harmlos. Aber:
- Zur Bewertung gehören meist ein **Name oder Nutzername** und eine **Kundennummer**.
- Im Text selbst stehen manchmal Namen, E-Mail-Adressen oder Telefonnummern.
- Und manchmal stehen dort Angaben, die die DSGVO besonders schützt (Art. 9, z. B. **Gesundheitsdaten**): „Ich trage sie mit meinem Hörgerät".

**Was passiert in der nächsten Zelle?** Unsere synthetischen Bewertungen enthalten nichts davon. Deshalb nehmen wir vier **ausgedachte** Beispiele und suchen mit einfachen Mustern nach E-Mail-Adressen und Telefonnummern.

In [ ]:
BEISPIELE = [
    "Super Klang! Bei Rückfragen gern an max.mustermann@example.com schreiben.",
    "Der Akku ist nach einer Stunde leer. Rufen Sie mich an: 0170 1234567",
    "Meine Frau Petra trägt sie zusammen mit ihrem Hörgerät, klappt gut.",
    "Bluetooth verbindet sofort, top verarbeitet.",
]

MUSTER = {
    "E-Mail": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "Telefon": r"(?:\+49|0)\s?\d{2,4}[\s/-]?\d{4,8}",
}

def finde_personenbezug(text):
    return {art: re.findall(m, text) for art, m in MUSTER.items() if re.findall(m, text)}

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({"Text": BEISPIELE, "gefunden": [finde_personenbezug(t) or "—" for t in BEISPIELE]}))

**Wie lese ich das?** E-Mail und Telefonnummer findet das Muster sofort. Den Namen „Petra" und den Hinweis auf ein **Hörgerät** findet es nicht. Namen und Gesundheitsangaben haben kein festes Muster.

Man könnte ein LLM fragen, ob ein Text personenbezogene Daten enthält. Aber dann hätte man die Daten ja schon an das LLM geschickt. Für genau solche Vorprüfungen sind **lokale** Modelle (Abschnitt 4) interessant: Die Prüfung findet statt, bevor etwas das Haus verlässt.

**Was passiert in der nächsten Zelle?** Was wir finden, ersetzen wir durch Platzhalter, **bevor** wir den Text verschicken. Das heißt *Pseudonymisieren*, und es folgt einem Grundsatz der DSGVO: nur so viele Daten verarbeiten, wie für den Zweck nötig sind (Datenminimierung, Art. 5 Abs. 1 lit. c).

In [ ]:
def pseudonymisiere(text):
    for art, m in MUSTER.items():
        text = re.sub(m, f"[{art.upper()}]", text)
    return text

for t in BEISPIELE:
    print(pseudonymisiere(t))

geaendert = sum(pseudonymisiere(t) != t for t in df["text"])
print(f"\nIn unseren {len(df)} synthetischen Bewertungen geändert: {geaendert}")

**Wie lese ich das?** E-Mail und Telefonnummer sind ersetzt, für die Analyse von Stimmung und Themen fehlt dadurch nichts. Name und Gesundheitsangabe stehen noch da: Einfache Muster reichen für eine echte Pseudonymisierung nicht aus. In unseren synthetischen Daten ändert sich nichts, weil der Generator keine persönlichen Angaben erzeugt. Das ist kein Zufall, sondern Absicht.

Ein angenehmer Nebeneffekt: Was man nicht verschickt, kostet auch keine Tokens.

### Die Fragen, bevor Kundentexte an eine US-API gehen

Diese Fragen stellt jedes Projekt, das echte Kundentexte an einen Cloud-Dienst schickt. Die Spalte rechts zeigt, was der Anbieter in unserem Fall **selbst dazu veröffentlicht** (Stand November 2026). Das ist eine Information, keine Bewertung, ob es für ein konkretes Projekt ausreicht.

| Frage | Worum es geht | Was Anthropic dazu veröffentlicht |
|---|---|---|
| **Gibt es eine Rechtsgrundlage?** (Art. 6) | Jede Verarbeitung personenbezogener Daten braucht eine, z. B. Einwilligung oder berechtigtes Interesse. | (Sache des Verantwortlichen, also des Shops) |
| **Wer ist wofür verantwortlich?** (Art. 28) | Wer Daten im Auftrag verarbeitet, braucht einen Vertrag zur Auftragsverarbeitung. | Für API-Kunden handelt Anthropic als **Auftragsverarbeiter** ([Quelle](https://privacy.claude.com/en/articles/9267385-does-anthropic-act-as-a-data-processor-or-controller)); ein Vertrag zur Auftragsverarbeitung (DPA) mit Standardvertragsklauseln ist Teil der Commercial Terms ([Quelle](https://privacy.claude.com/en/articles/7996862-how-do-i-view-and-sign-your-data-processing-addendum-dpa)). |
| **Dürfen die Daten in die USA?** (Art. 44 ff.) | Übermittlung in ein Drittland nur mit Angemessenheitsbeschluss oder geeigneten Garantien. | Der EU-US Data Privacy Framework gilt seit dem 10.07.2023, aber nur für US-Organisationen, die in der offiziellen Liste stehen ([Quelle](https://www.edpb.europa.eu/system/files/2024-12/edpb_informationnoteadequacydecisionus_de.pdf), [Liste](https://www.dataprivacyframework.gov/s/participant-search)). Ob ein Anbieter gelistet ist, prüft man dort selbst. |
| **Werden die Daten zum Training genutzt?** | Fließen Kundentexte in künftige Modelle? | Laut Anthropic werden aufbewahrte Daten nicht ohne ausdrückliche Erlaubnis zum Training genutzt ([Quelle](https://platform.claude.com/docs/en/manage-claude/api-and-data-retention)). |
| **Wie lange werden sie gespeichert?** | Wo liegen die Daten, wie lange, wer kann sie sehen? | Hängt von Funktion und Vertrag ab; die Doku listet das je Funktion ([Quelle](https://platform.claude.com/docs/en/manage-claude/api-and-data-retention)). |
| **Sind besondere Daten dabei?** (Art. 9) | Gesundheit, Religion u. a. sind besonders geschützt. | (Sache des Verantwortlichen: vorher prüfen und minimieren) |

Ein paar Dinge in dieser Tabelle sind leicht zu übersehen:
- **Verantwortlich bleibt der Shop**, nicht der LLM-Anbieter. Ein Vertrag mit dem Anbieter nimmt einem die eigenen Pflichten nicht ab.
- **Veröffentlichte Regeln ändern sich.** Alles in der rechten Spalte gilt für den Zeitpunkt, an dem man nachschaut, und muss bei einem echten Projekt neu geprüft werden.
- Über einen Cloud-Anbieter wie AWS oder Google Cloud gelten wieder andere Verträge ([Quelle](https://platform.claude.com/docs/en/manage-claude/api-and-data-retention)).

**Was passiert in der nächsten Zelle?** Die Fragen als Checkliste in Code, angewendet auf unseren Kurs. So kann man sie für jedes Projekt wiederverwenden.

In [ ]:
DSGVO_FRAGEN = [
    ("Enthalten die Texte personenbezogene Daten?", "personenbezug"),
    ("Gibt es eine Rechtsgrundlage für die Verarbeitung (Art. 6)?", "rechtsgrundlage"),
    ("Gibt es einen Vertrag zur Auftragsverarbeitung mit dem Anbieter (Art. 28)?", "avv"),
    ("Ist die Übermittlung in die USA abgesichert (Art. 44 ff.)?", "drittland"),
    ("Werden die Daten vor dem Senden minimiert / pseudonymisiert (Art. 5)?", "minimierung"),
    ("Sind besondere Kategorien ausgeschlossen oder geprüft (Art. 9)?", "art9"),
]

def checkliste(antworten):
    zeilen = []
    for frage, schluessel in DSGVO_FRAGEN:
        a = antworten.get(schluessel, "offen")
        gut = "nein" if schluessel == "personenbezug" else "ja"      # bei Frage 1 ist „nein" die gute Antwort
        zeichen = "➖" if a == "entfällt" else "✅" if a == gut else "❓" if a == "offen" else "⚠️"
        zeilen.append({"Frage": frage, "Antwort": f"{zeichen} {a}"})
    return pd.DataFrame(zeilen).set_index("Frage")

kurs = {"personenbezug": "nein", "rechtsgrundlage": "entfällt", "avv": "entfällt",
        "drittland": "entfällt", "minimierung": "entfällt", "art9": "entfällt"}
with pd.option_context("display.max_colwidth", None):
    display(checkliste(kurs))

**Wie lese ich das?** Für unseren Kurs entfällt fast alles, weil die erste Antwort „nein" ist: synthetische Daten, keine echten Personen. Das ist der zweite Grund für den synthetischen Datensatz, neben der Lizenz. Bei echten Daten verwandelt sich jedes „entfällt" in eine Frage, die jemand beantworten und dokumentieren muss. Und ❓ heißt nicht „Problem", sondern „noch niemand hat nachgeschaut".

> **Nicht:** „Die Bewertungen sind sowieso öffentlich, also dürfen wir damit alles machen."
> **Sondern:** Öffentlich lesbar heißt weder lizenzfrei noch datenschutzfrei. Vor dem ersten echten Lauf: Rechte klären, Daten minimieren, und mit dem oder der Datenschutzbeauftragten sprechen.

## 7. Eure Aufgabe: euer eigenes Projekt

✏️ Für die Projektarbeit wählt ihr eigene Daten. Beschreibt euer Projekt (oder eine Idee dafür) in einem kleinen Dictionary. Die Funktion `projekt_check` in der nächsten Zelle macht daraus eine Kostenschätzung und die Fragen, die ihr klären müsst.

1. Führt die nächste Zelle aus (sie ist fertig, keine Lücken).
2. Füllt in der Zelle danach `mein_projekt` mit euren Angaben.
3. Notiert **eine Lizenzfrage** und **eine Datenschutzfrage**, die ihr vor dem ersten echten Lauf klären müsst.

In [ ]:
def projekt_check(p):
    """Kosten, Dauer und offene Rechtsfragen für ein Projekt. Faustregeln, keine Rechtsprüfung."""
    k = kosten(p["anzahl_texte"], p["tokens_pro_text"] + tok_prompt, TOK_OUT)
    print(f"Projekt: {p['name']}")
    print(f"  Kosten einmalig:        ca. {k:,.2f} USD ({p['anzahl_texte']:,} Texte)".replace(",", "."))
    print(f"  Dauer bei 8 parallel:   ca. {dauer_text(p['anzahl_texte'] * LATENZ / 8)}")
    print("\n  Lizenzfragen:")
    if not p["eigene_daten"]:
        print("   - Wer hat die Rechte an den Daten, und unter welchen Bedingungen dürfen wir sie nutzen?")
    if p["kommerziell"]:
        print("   - Erlauben die Bedingungen der Quelle eine kommerzielle Nutzung?")
    if p["veroeffentlichen"]:
        print("   - Dürfen wir die Daten (oder Ausschnitte) weitergeben, z. B. in ein öffentliches Repository?")
    print("   - Erlauben die Bedingungen, die Texte an einen KI-Dienst zu schicken?")
    print("\n  Datenschutzfragen:")
    if p["personenbezug_moeglich"]:
        print("   - Welche personenbezogenen Daten enthalten die Texte, und können wir sie vorher entfernen?")
        print("   - Auf welcher Rechtsgrundlage verarbeiten wir sie, und wer ist verantwortlich?")
        print(f"   - Die Daten gehen an {p['anbieter']}: Vertrag zur Auftragsverarbeitung? Übermittlung in Drittländer?")
    else:
        print("   - Sicherstellen und dokumentieren, dass wirklich keine personenbezogenen Daten enthalten sind.")
    print("\n  Ansprechperson für alle Rechtsfragen: Datenschutzbeauftragte/r (HTW bzw. Unternehmen).")

In [ ]:
# Eure Lösung: Lücken ... füllen
# (try/except nur, damit „Alle ausführen" hier nicht stoppt. Die Lösungszelle darunter braucht es nicht.)
try:
    mein_projekt = {
        "name": ...,                         # z. B. "Hotelbewertungen Berlin"
        "anzahl_texte": ...,                 # wie viele Texte wollt ihr analysieren?
        "tokens_pro_text": ...,              # grob: Zeichen pro Text geteilt durch 4
        "eigene_daten": ...,                 # True / False
        "kommerziell": ...,                  # True / False: ist ein Unternehmen beteiligt?
        "veroeffentlichen": ...,             # True / False: kommen Daten ins Repository?
        "personenbezug_moeglich": ...,       # True / False
        "anbieter": ...,                     # z. B. "Anthropic (USA)"
    }
    projekt_check(mein_projekt)
except Exception as e:
    print("Noch nicht fertig oder ein Fehler:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Lösung (ein Beispiel): erst selbst versuchen, dann ausführen und vergleichen.
beispiel_projekt = {
    "name": "Hotelbewertungen von einer Buchungsplattform",
    "anzahl_texte": 50_000,
    "tokens_pro_text": 150,
    "eigene_daten": False,
    "kommerziell": False,
    "veroeffentlichen": True,
    "personenbezug_moeglich": True,
    "anbieter": "Anthropic (USA)",
}
projekt_check(beispiel_projekt)

**Wie lese ich das?** Die Kosten sind fast immer die kleinste Frage. Im Beispiel kostet die Analyse von 50.000 Hotelbewertungen einen überschaubaren Betrag, aber drei Lizenzfragen und drei Datenschutzfragen sind offen. Bei der Buchungsplattform ist die wichtigste davon die erste: Wer hat die Rechte an den Bewertungen, und darf man sie für ein Projekt abrufen und speichern?

Für eure Projektarbeit gilt: Wählt Daten, deren Lizenz ihr **nachweisen** könnt (offene Lizenzen wie CC, eigene Daten oder synthetische Daten), und schreibt in eure Präsentation, wie ihr die beiden Fragen beantwortet habt.

## 8. Der Bericht: Was würde es kosten, und was ist zu klären?

Wie in Sitzung 7 fassen wir alles für Lena auf einer Seite zusammen.

**Was passiert in der nächsten Zelle?** Ein kurzer Bericht aus den Zahlen von heute. Alle Zahlen kommen aus den Variablen oben.

In [ ]:
bericht = f"""
## Review Radar im Großen: Kosten, Dauer, offene Fragen

**Grundlage:** {len(messung)} gemessene Analysen mit Claude Haiku 4.5: im Mittel {TOK_IN:.0f} Input- und {TOK_OUT:.0f} Output-Tokens,
{LATENZ:.1f} Sekunden pro Aufruf.

**Kosten:** {kosten(1, TOK_IN, TOK_OUT) * 1000:.2f} USD pro 1.000 Bewertungen. Für alle 40 Produkte:
einmalig {einmalig:,.0f} USD für den Bestand, danach {pro_monat:,.0f} USD pro Monat.
Mit Batching und Batch-API sind etwa {hebel.iloc[-1] / hebel.iloc[0]:.0%} davon erreichbar, wenn die Qualität gehalten wird.

**Dauer:** Nacheinander {dauer_text(bestand * LATENZ)} für den Bestand; am Rate-Limit ({engpass}) {dauer_text(bestand / max_pro_minute * 60)}.

**Vor dem ersten echten Lauf zu klären:**
1. Rechte an den Bewertungen: Dürfen wir sie für diesen Zweck auswerten und an einen KI-Dienst schicken?
2. Datenschutz: Personenbezug minimieren, Rechtsgrundlage und Vertrag zur Auftragsverarbeitung prüfen,
   Übermittlung in die USA absichern. Ansprechperson: Datenschutzbeauftragte/r.

*Keine Rechtsberatung: Die Fragen zeigen, was zu klären ist, nicht wie die Antwort lautet.*
"""
display(Markdown(bericht))

**Wie lese ich das?** Zwei Absätze Zahlen, ein Absatz Fragen. Das ist die Antwort auf die Frage vom Anfang: *Was würde es kosten, und was ist rechtlich zu klären, um es wirklich zu betreiben?* Die Zahlen kann man ausrechnen. Die Fragen kann man nur stellen und von den richtigen Leuten beantworten lassen. Beides gehört in jede Projektpräsentation.

## Zusammenfassung

```text
1. Was kostet es?                      -> Tokens messen (rein, raus), mal Preis, mal Anzahl
2. Wofür bezahlen wir?                 -> vor allem Prompt und Antwort, kaum die Daten selbst
3. Wie wird es billiger?               -> Batching, Batch-API, kürzerer Prompt; jede Ersparnis neu messen
4. Wie lange dauert es?                -> nacheinander: Tage; parallel: bis zum Rate-Limit (Little's Law)
5. Lokal oder Cloud?                   -> Kosten selten entscheidend; oft entscheiden die Daten
6. Darf ich die Daten nutzen?          -> Lizenz und Nutzungsbedingungen; öffentlich != frei
7. Darf ich sie verschicken?           -> DSGVO: Personenbezug, Rechtsgrundlage, Auftragsverarbeitung,
                                          Drittland; Datenschutzbeauftragte/r fragen
```

## Zum Weiterprobieren (freiwillig)

1. **Ein teureres Modell:** Setzt in `kosten` die Preise eines größeren Modells aus der Preisliste von Anthropic ein. Wie verändert sich Lenas Rechnung?
2. **Der Batch-Prompt:** Nehmt den Batch-Prompt aus Sitzung 3, messt Input- und Output-Tokens für 20 Bewertungen pro Aufruf und vergleicht mit der Schätzung in Abschnitt 2.
3. **Andere Grenzen:** Was ändert sich am Engpass, wenn die Antworten doppelt so lang werden (z. B. eine längere Begründung)?
4. **Besseres Pseudonymisieren:** Erweitert `MUSTER` um weitere Muster (IBAN, Postleitzahl mit Ort). Was bleibt trotzdem unentdeckt?

## Ausblick

Das war die letzte Sitzung vor der Weihnachtspause. Phase 3 hat gezeigt, wo ein funktionierender Prototyp an der Realität scheitern kann: am lügenden LLM, an schlechten Daten, an Kosten und Recht.

**Nächste Sitzung (06.01.2027, online):** *Woher weiß man, dass die Analyse stimmt?* Wir annotieren selbst eine Stichprobe und messen das Modell daran, der Schritt, den ihr für eure Projektarbeit braucht.

**Quellen (abgerufen am 18.11.2026):**
- Anthropic, Preise der Claude API: https://platform.claude.com/docs/en/about-claude/pricing
- Anthropic, Batch-Verarbeitung: https://platform.claude.com/docs/en/build-with-claude/batch-processing
- Anthropic, API und Datenaufbewahrung: https://platform.claude.com/docs/en/manage-claude/api-and-data-retention
- Anthropic, Data Processing Addendum: https://privacy.claude.com/en/articles/7996862-how-do-i-view-and-sign-your-data-processing-addendum-dpa
- Anthropic, Auftragsverarbeiter oder Verantwortlicher: https://privacy.claude.com/en/articles/9267385-does-anthropic-act-as-a-data-processor-or-controller
- Multilingual Amazon Reviews Corpus, Nutzungsbedingungen: https://docs.opendata.aws/amazon-reviews-ml/readme.html
- DSGVO (Verordnung (EU) 2016/679): https://eur-lex.europa.eu/eli/reg/2016/679/oj/deu
- EDSA, Informationsvermerk zum EU-US-Datenschutzrahmen: https://www.edpb.europa.eu/system/files/2024-12/edpb_informationnoteadequacydecisionus_de.pdf
- Data Privacy Framework List: https://www.dataprivacyframework.gov/s/participant-search
- Creative Commons BY 4.0: https://creativecommons.org/licenses/by/4.0/